# week16 · 01 큐레이션 → 파레토 → EHVI
덱 5 · 7 · 9 · 11 · 14장. 순서: curate.py(672 → 612) → pareto.py(전선 · HV) → ehvi_select.py(다음 MD 3건). 전부 CPU · 약 30초.

**기본 경로 = 로컬 mdai + Claude Code** · week16 폴더에서 `claude` → 덱 7 · 14장 지시 P16-1 · P16-2 가 같은 스크립트를 실행한다(`../prompts.md`). 첫 셀은 Colab(선택 경로)용이다.

In [ ]:
# ① 저장소 받기 · Colab(선택 경로)용 · 로컬(mdai · Claude Code)은 week16 폴더에서 시작하므로 건너뜀
import sys
if "google.colab" in sys.modules:   # (선택) Colab 일 때만 · 로컬(Claude Code · run_nb.py)은 주차 폴더에서 바로 실행
    !git clone https://github.com/sentirain7/2026_MOOC_Lecture_Materials.git 2>/dev/null
    %cd */week16
    !pip -q install scikit-learn numpy

In [ ]:
# ② 큐레이션 깔때기 (덱 5 · 7장) — 지운 행 0 · 단계별 행 수 · 이유
!python3 scripts/curate.py data/core672.csv --fixed data/features614.csv --out out/curation.json --csv out/curated612.csv
import json
for s in json.load(open('out/curation.json'))['stages']: print(s)

In [ ]:
# ③ 파레토 전선 · 초부피 (덱 9장) — 참조점을 바꿔 HV 가 달라지는지 확인
!python3 scripts/pareto.py out/curated612.csv --ref 1.0,250 --out out/pareto.json
p = json.load(open('out/pareto.json')); print('전선', len(p['front']), '· HV', p['hv'], '· 참조점 (3, 350) HV', p['hv_alt_ref']['hv'])
for f in p['front']: print(f)

In [ ]:
# ④ EHVI 배치 3개 (덱 11 · 14장) — 약 17초 · ConvergenceWarning(길이척도 상한) 은 limits.md 의 한계 1
!python3 scripts/ehvi_select.py data/features614.csv out/curated612.csv --obs 400 --batch 3 --out out/ehvi.json
e = json.load(open('out/ehvi.json'))
for k in ['hv_obs', 'hv_after_true', 'hv_gain_ehvi', 'hv_gain_random_mean', 'hv_gain_random_p_ge']: print(k, e[k])
for r in e['picks']: print(r['rank'], r['binder'], r['additive'], r['aging'], r['T_K'], 'μ', r['mu'], 'σ', r['sd'], '실제', r['true'])

In [ ]:
# ⑤ 그림 — 612점 · 전선 · EHVI 선택 (덱 9 · 11장과 같은 축)
import matplotlib.pyplot as plt
P = p['points']; F = sorted([(f['K'], f['CED']) for f in p['front']])
plt.figure(figsize=(6.5, 4.2)); plt.scatter([q[0] for q in P], [q[1] for q in P], s=6, c='#B9AE9E')
plt.step([f[0] for f in F], [f[1] for f in F], where='pre', c='#E37F1A'); plt.scatter(*zip(*F), c='#E37F1A', zorder=3)
for r in e['picks']: plt.scatter(*r['true'], marker='s', c='k', zorder=4)
plt.xlabel('K (GPa)'); plt.ylabel('CED (MJ/m³)'); plt.show()

**바꿔 보기**: `pareto.py --ref 3.0,350` (덱 9장 ③) · `ehvi_select.py --obs 300 --batch 5 --seed 11` · 목표 열 하나만 남기면 전선이 1점이 되는지(덱 13장). 결과 파일은 P16-1 · P16-2 Claude Code 지시가 직접 읽는다(`../prompts.md`).